# Replace `ops_pipeline3` with Spark

Metadata-driven incremental ingestion from an Eventhouse (KQL database) into Lakehouse Delta tables.

This notebook replaces the pipeline's two steps:
1. **LookupDueJobs** → read the control table `dbo.datacopyjobsetup`.
2. **ForEach + CopyDynamicKQLTables** → for each job, run an incremental KQL query and append the result to `dbo.<DestinationName>`.

Each control row provides: `SourceName`, `WatermarkColumn`, `LastUpdated`, `DestinationName`.
The KQL query built per job is: `<SourceName> | where <WatermarkColumn> > datetime(<LastUpdated>)`.

## Parameters

In [ ]:
# Pipeline parameters translated to notebook parameters.
kql_cluster = "https://trd-7qm6ccfqm2rr8uzwff.z0.kusto.fabric.microsoft.com"
kql_database = ""  # was pipeline parameter ops_kql_db

config_table = "datacopyjobsetup"  # control table read by LookupDueJobs
dest_schema = "dbo"                 # sink schema in the lakehouse
max_parallel = 8                    # mirrors the pipeline ForEach batchCount (20)
advance_watermark = False           # pipeline does NOT advance the watermark; keep off to match it

## Setup: Kusto token and helpers

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

import notebookutils

# AAD token for the Eventhouse/Kusto cluster, using the notebook's identity.
kusto_token = notebookutils.credentials.getToken(kql_cluster)


def read_kql(query: str):
    """Run a KQL query against the Eventhouse and return a Spark DataFrame."""
    return (
        spark.read.format("com.microsoft.kusto.spark.datasource")
        .option("kustoCluster", kql_cluster)
        .option("kustoDatabase", kql_database)
        .option("kustoQuery", query)
        .option("accessToken", kusto_token)
        .load()
    )


def build_query(source_name: str, watermark_column: str, last_updated) -> str:
    """Reproduce the pipeline's dynamic KQL: <source> | where <col> > datetime(<iso>)."""
    iso = last_updated.strftime("%Y-%m-%dT%H:%M:%SZ") if last_updated is not None else "1900-01-01T00:00:00Z"
    return f"{source_name} | where {watermark_column} > datetime({iso})"

## Step 1 — LookupDueJobs (read the control table)

In [ ]:
jobs = spark.table(f"{dest_schema}.{config_table}").collect()
print(f"Loaded {len(jobs)} copy job(s) from {dest_schema}.{config_table}")
for j in jobs:
    print(f"  - {j['SourceName']} -> {dest_schema}.{j['DestinationName']} (watermark {j['WatermarkColumn']} > {j['LastUpdated']})")

## Step 2 — ForEach + Copy (incremental KQL → Lakehouse append)

In [ ]:
def run_job(job) -> dict:
    source_name = job["SourceName"]
    watermark_column = job["WatermarkColumn"]
    last_updated = job["LastUpdated"]
    destination = job["DestinationName"]

    query = build_query(source_name, watermark_column, last_updated)
    df = read_kql(query)
    row_count = df.count()

    if row_count > 0:
        (
            df.write.mode("append")
            .format("delta")
            .option("mergeSchema", "true")
            .saveAsTable(f"{dest_schema}.{destination}")
        )

    return {"source": source_name, "destination": destination, "rows": row_count, "query": query}


results = []
with ThreadPoolExecutor(max_workers=max_parallel) as pool:
    futures = {pool.submit(run_job, j): j for j in jobs}
    for fut in as_completed(futures):
        job = futures[fut]
        try:
            res = fut.result()
            print(f"OK   {res['source']} -> {dest_schema}.{res['destination']}: {res['rows']} row(s) appended")
            results.append(res)
        except Exception as exc:  # noqa: BLE001 - report per-job failure, continue others
            print(f"FAIL {job['SourceName']} -> {job['DestinationName']}: {exc}")
            results.append({"source": job["SourceName"], "destination": job["DestinationName"], "error": str(exc)})

print(f"\nCompleted {len(results)} job(s).")

## Optional — advance the watermark

The original pipeline never updated `LastUpdated`, so re-runs re-copy the same rows. Enable this to make the run truly incremental by advancing each job's watermark to the max value just ingested.

In [ ]:
from pyspark.sql import functions as F

if advance_watermark:
    from delta.tables import DeltaTable

    control = DeltaTable.forName(spark, f"{dest_schema}.{config_table}")
    for res in results:
        if res.get("rows", 0) and "error" not in res:
            job = next(j for j in jobs if j["DestinationName"] == res["destination"])
            new_max = (
                spark.table(f"{dest_schema}.{res['destination']}")
                .agg(F.max(job["WatermarkColumn"]).alias("m"))
                .collect()[0]["m"]
            )
            if new_max is not None:
                control.update(
                    condition=F.col("DestinationName") == res["destination"],
                    set={"LastUpdated": F.lit(new_max)},
                )
                print(f"Watermark for {res['destination']} advanced to {new_max}")
else:
    print("advance_watermark is False — matching original pipeline behavior (watermark unchanged).")